In [4]:
# 1. Environment setup: clone before installing; safe to rerun.
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import os
import platform
import subprocess
import sys

REPO_URL = 'https://github.com/jcatumba/neonatal-sepsis-lstm.git'
REPO_REF = 'main'  # Set a release tag or commit here to reproduce a code revision.
IN_COLAB = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ
BASE_DIR = Path('/content') if IN_COLAB else Path.cwd()
REPO_DIR = BASE_DIR / 'neonatal-sepsis-lstm' if IN_COLAB else BASE_DIR
if not (REPO_DIR / 'requirements-colab.txt').is_file():
    if REPO_DIR.exists():
        raise FileNotFoundError(f'{REPO_DIR} exists but lacks requirements-colab.txt. Use a fresh runtime or the project root locally.')
    subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_DIR)])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
if not (3, 11) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError('Use a Python 3.11-3.13 Colab runtime for the pinned dependencies.')
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', '-r', str(REPO_DIR / 'requirements-colab.txt')])
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('Python:', platform.python_version())
for package in ('torch', 'tensorflow'):
    try:
        print(f'{package}: {version(package)}')
    except PackageNotFoundError:
        print(f'{package}: not installed (TensorFlow is not required)')
print('Code revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


Python: 3.13.16
torch: 2.8.0
tensorflow: 2.21.0
Code revision: b17baf6bd649958305871fe087f3634f5a4570db


# Neonatal sepsis LSTM autoencoder
Choose **Runtime > Run all**. Defaults run a small, explicitly synthetic demo without uploads or Drive authorization. Optional: select a T4 GPU before running.

**Research demonstration only, not a medical device or diagnostic tool.** Synthetic metrics do not measure clinical performance. For private data, use an approved environment, de-identify records, and do not publish notebook outputs. Dependencies are pinned; save the printed Git revision for reproducibility. If packages were imported before setup, restart the runtime and Run all.

## 2. Imports and configuration
Switch `DATA_SOURCE` to `upload`, `drive`, or `url` for your CSV. Leave `demo` for a no-edit Run all. Drive/upload modes require interaction. Set `FAST_DEV_RUN=False` for full training. CPU fallback is automatic.

In [5]:
import json
import torch
from IPython.display import display, Image
from src.data import demo_data, load_csv, prepare_data, FEATURE_COLUMNS
from src.model import LSTMAutoencoder
from src.train import train_model
from src.eval import evaluate_model
from src.pipeline import save_artifacts, predict_frame
from src.utils import Config, set_seed, save_to_drive_if_enabled

FAST_DEV_RUN = True
DATA_SOURCE = 'demo'  # demo | upload | drive | url
DRIVE_CSV = Path('/content/drive/MyDrive/neonatal/data.csv')
CSV_URL = ''  # Public HTTPS CSV; never include private credentials.
COPY_TO_DRIVE = False
config = Config(
    seed=42, batch_size=32, seq_len=24, latent_dim=16, hidden_dim=32,
    epochs=20, learning_rate=1e-3, device_preference='cuda',
    fast_dev_run=FAST_DEV_RUN,
    output_dir=BASE_DIR / 'outputs' / 'neonatal-sepsis',
    drive_output_dir=Path('/content/drive/MyDrive/neonatal/outputs') if COPY_TO_DRIVE else None,
)
set_seed(config.seed)
print('Device:', config.device, '| fast dev run:', FAST_DEV_RUN)


ModuleNotFoundError: No module named 'src'

## 3. Data loading and validation
CSV: `patient_id`, ISO-8601 `timestamp`, binary `sepsis_label`, plus the eight feature columns listed below. One row per regularly sampled time point; consistent cadence/units across patients. At least five entirely normal and five sepsis patients, each with at least `seq_len` rows. Labels describe each row; a window is positive if any row is positive. No automatic download of clinical data.

Missing values and implausible measurements are interpolated within each patient only. An entirely missing feature is rejected. We do not resample or infer clinical labels.

In [ ]:
if DATA_SOURCE not in {'demo', 'upload', 'drive', 'url'}:
    raise ValueError('Choose demo, upload, drive, or url.')
if DATA_SOURCE == 'drive' or COPY_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
if DATA_SOURCE == 'demo':
    print('SYNTHETIC DEMO ONLY: metrics have no clinical validity.')
    frame = demo_data(config.seed, config.seq_len)
elif DATA_SOURCE == 'upload':
    from google.colab import files
    import io
    import pandas as pd
    from src.data import validate_frame
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one CSV file.')
    filename, content = next(iter(uploaded.items()))
    frame = validate_frame(pd.read_csv(io.BytesIO(content)))
elif DATA_SOURCE == 'drive':
    frame = load_csv(DRIVE_CSV)
else:
    if not CSV_URL.startswith('https://'):
        raise ValueError('Set CSV_URL to a public HTTPS CSV URL.')
    frame = load_csv(CSV_URL)
print('Features:', FEATURE_COLUMNS)
print('Rows:', len(frame), '| patients:', frame.patient_id.nunique())


## 4. Preprocessing and sequences
Stratified 60/20/20 patient-disjoint splits precede windowing. Training and validation use entirely normal patients only. Fit MinMax scaling on training windows only; use normal validation errors for threshold calibration. Test data never tune the threshold. Windows are nonoverlapping and never cross patients. Smoke mode caps sequences after splitting (32 per class).

In [ ]:
data = prepare_data(frame, config)
print('Train:', data.train.shape, 'Validation:', data.validation.shape, 'Test:', data.test.shape)


## 5. Build the original LSTM encoder/decoder

In [ ]:
model = LSTMAutoencoder(
    input_dim=len(FEATURE_COLUMNS), hidden_dim=config.hidden_dim,
    latent_dim=config.latent_dim, seq_len=config.seq_len,
    num_layers=config.num_layers, dropout=config.dropout,
)
print('Parameters:', sum(p.numel() for p in model.parameters()))


## 6. Train
At most two epochs in smoke mode. Full mode uses configured epochs with early stopping, gradient clipping and learning-rate scheduling. Reloads the best validation checkpoint before evaluation.

In [ ]:
history = train_model(model, data.train, data.validation, config)
display(Image(filename=str(config.output_dir / 'training_curve.png')))


## 7. Evaluate and select threshold
Threshold = configured percentile of normal **validation** MSE. Report window-level held-out AUROC, average precision (AUPRC), sensitivity and specificity. Overlapping clinical outcomes, prevalence and sampling affect these metrics; they are not patient-level clinical validation.

In [ ]:
metrics, test_errors = evaluate_model(model, data.validation, data.test, data.test_labels, config)
print(json.dumps(metrics, indent=2))
for plot in ('roc_curve.png', 'pr_curve.png'):
    display(Image(filename=str(config.output_dir / plot)))


## 8. Persist artifacts
Colab outputs are under `/content/outputs/neonatal-sepsis`. Drive copy is opt-in. Local outputs are overwritten on reruns; choose a different `output_dir` to retain each experiment. No raw data or patient IDs are exported by the artifact helper. Preprocessor files use pickle: load only your own trusted artifacts.

In [ ]:
save_artifacts(data, config)
save_to_drive_if_enabled(config)
print('Artifacts:', sorted(path.name for path in config.output_dir.iterdir()))


## 9. Inference on new sample
Reload model, preprocessor and threshold from disk. The default example is a newly generated synthetic patient, not a test patient. Replace `new_sample` with `load_csv(Path(...))` for new CSV observations using identical cadence, schema and units. For unlabeled new data, supply `sepsis_label=0` as a placeholder; prediction does not use it. Anomaly flags are **not diagnoses**.

In [ ]:
new_sample = demo_data(config.seed + 1, config.seq_len)
new_sample = new_sample[new_sample.patient_id == 'synthetic-000'].copy()
new_sample['patient_id'] = 'new-synthetic-patient'
prediction = predict_frame(new_sample, config.output_dir, batch_size=config.batch_size)
display(prediction.drop(columns='patient_id'))
